# Input/Output & External Data Sources

read and write CSV files;

inspect data after ingestion;

work with project-relative paths;

create and query an SQLite database;

load SQL results into pandas;

retrieve data from an HTTP API;

inspect status codes;

parse and normalize JSON;

read HTML tables;

save and load Parquet;

understand the role of HDF5;

validate externally sourced data;

preserve raw data and record provenance.

In [1]:
from pathlib import Path
import sqlite3
import json

import pandas as pd
import requests

In [2]:
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

In [3]:
participants = pd.DataFrame({
    "id": ["001", "002", "003", "004", "005"],
    "age": [24, 31, 27, 42, 35],
    "group": ["A", "B", "A", "B", "A"],
    "score": [81, 94, 88, 91, 85]
})

In [4]:
print(f"{participants = }")
print(f"{participants.dtypes = }")

participants =     id  age group  score
0  001   24     A     81
1  002   31     B     94
2  003   27     A     88
3  004   42     B     91
4  005   35     A     85
participants.dtypes = id         str
age      int64
group      str
score    int64
dtype: object


Storing a string as an ID means we don't loose the leading zeroes, rather than an int which would collapse ID `001` to 1.

In [5]:
csv_path = RAW_DIR / "participants.csv"

participants.to_csv(
    csv_path,
    index=False
)

In [6]:
csv_path.exists()

True

There is no data-type information stored explicitly in the CSV.

In [7]:
loaded = pd.read_csv(csv_path)
loaded.dtypes

id       int64
age      int64
group      str
score    int64
dtype: object

ID was read as an integer, so the value collapsed the leading zeroes to 1.

In [8]:
loaded = pd.read_csv(
    csv_path,
    dtype={
        "id": "string",
        "group": "category"
    }
)
print(f"Types: \n{loaded.dtypes}") 
loaded

Types: 
id         string
age         int64
group    category
score       int64
dtype: object


,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


An identifier is often better represented as a string than as a numeric measurement because it does not numerically measure anything. Rather, it is more of a categorical.

In [9]:
messy = pd.DataFrame({
    "id": ["001", "002", "003", "004"],
    "score": ["81", "-999", "88", "NA"]
})

messy_path = RAW_DIR / "messy_scores.csv"

messy.to_csv(
    messy_path,
    index=False
)

In [10]:
messy_loaded = pd.read_csv(
    messy_path,
    na_values=["NA", "-999"]
)

In [11]:
print(f"{messy_loaded}\n")
print(f"dtypes\n{messy_loaded.dtypes}\n")
print(f"NAs\n{messy_loaded.isna().sum()}\n")

   id  score
0   1   81.0
1   2    NaN
2   3   88.0
3   4    NaN

dtypes
id         int64
score    float64
dtype: object

NAs
id       0
score    2
dtype: int64



Documenting missing-value coding is part of provenance because it helps to explain why values are missing and identify if it is systemic, intentional, or truly random among other reasons.

In [12]:
print(f".shape: \n{loaded.shape}\n")
print(f"head(): \n{loaded.head()}\n")
print(f".dtypes: \n{loaded.dtypes}\n")
print(f"sum(): \n{loaded.isna().sum()}\n")
print(f"nunique(): \n{loaded["id"].nunique()}\n")
print(f"value_counts(): \n{loaded["group"].value_counts()}\n")

.shape: 
(5, 4)

head(): 
    id  age group  score
0  001   24     A     81
1  002   31     B     94
2  003   27     A     88
3  004   42     B     91
4  005   35     A     85

.dtypes: 
id         string
age         int64
group    category
score       int64
dtype: object

sum(): 
id       0
age      0
group    0
score    0
dtype: int64

nunique(): 
5

value_counts(): 
group
A    3
B    2
Name: count, dtype: int64



In [13]:
assert len(loaded) == 5
assert loaded["id"].is_unique
assert set(
    loaded["group"].astype(str)
) <= {"A", "B"}

# SQL

In [14]:
db_path = RAW_DIR / "study.db"

connection = sqlite3.connect(db_path)

In [15]:
participants.to_sql(
    "participants",
    connection,
    if_exists="replace",
    index=False
)

5

In [16]:
sql_df = pd.read_sql(
    "SELECT * FROM participants",
    connection
)

sql_df

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


What role did SQL play, and what role did pandas play?

SQL selected the values and pandas gave the SQL connection the selection query to run.

In [17]:
query = """
SELECT
    id,
    age,
    score
FROM participants
"""

selected = pd.read_sql(
    query,
    connection
)

selected

,id,age,score
0,001,24,81
1,002,31,94
2,003,27,88
3,004,42,91
4,005,35,85


In [18]:
participants[
    ["id", "age", "score"]
]

,id,age,score
0,001,24,81
1,002,31,94
2,003,27,88
3,004,42,91
4,005,35,85


In [19]:
query = """
SELECT *
FROM participants
WHERE age >= 30
"""

older = pd.read_sql(
    query,
    connection
)
older

,id,age,group,score
0,002,31,B,94
1,004,42,B,91
2,005,35,A,85


In [20]:
participants.loc[
    participants["age"] >= 30
]

,id,age,group,score
1,002,31,B,94
3,004,42,B,91
4,005,35,A,85


Filtering with SQL could be advantageous for a large database because it filters the data before loading, so less local memory space is needed.

In [21]:
query = """
SELECT
    "group" as group_name,
    COUNT(*) AS n,
    AVG(score) AS mean_score
FROM participants
GROUP BY "group"
"""

# query = """
# SELECT
#     group,
#     COUNT(*) AS n,
#     AVG(score) AS mean_score
# FROM participants
# GROUP BY group
# """

summary = pd.read_sql(
    query,
    connection
)

summary

,group_name,n,mean_score
0,A,3,84.666667
1,B,2,92.500000


In [22]:
query = """
SELECT
    "group" as group_name,
    MAX(score) as max_score,
    MIN(score) AS min_score
FROM participants
GROUP BY "group"
"""

minmax_summary = pd.read_sql(
    query,
    connection
)

minmax_summary

,group_name,max_score,min_score
0,A,88,81
1,B,94,91


In [23]:
query = """
SELECT *
FROM participants
WHERE "group" = ?
"""

group_a = pd.read_sql(
    query,
    connection,
    params=("A",)
)
group_a

,id,age,group,score
0,001,24,A,81
1,003,27,A,88
2,005,35,A,85


Parameters maight be preferable to inserting values directly into a query string so that different funcitons can tailor a standard query structure to their needs, or grab different subsets using the same query.

In [24]:
outcomes = pd.DataFrame({
    "id": ["001", "002", "003", "005"],
    "followup_score": [85, 97, 91, 89]
})

outcomes.to_sql(
    "outcomes",
    connection,
    if_exists="replace",
    index=False
)

4

In [25]:
# query = """
# SELECT
#     p.id,
#     p.age,
#     p.group,
#     p.score,
#     o.followup_score
# FROM participants AS p
# LEFT JOIN outcomes AS o
#     ON p.id = o.id
# """

query = """
SELECT
    p.id,
    p.age,
    p."group",
    p.score,
    o.followup_score
FROM participants AS p
LEFT JOIN outcomes AS o
    ON p.id = o.id
"""

joined = pd.read_sql(
    query,
    connection
)

In [26]:
print(joined)
print(f"\n\nNAs: \n {joined.isna().sum()}")

    id  age group  score  followup_score
0  001   24     A     81            85.0
1  002   31     B     94            97.0
2  003   27     A     88            91.0
3  004   42     B     91             NaN
4  005   35     A     85            89.0


NAs: 
 id                0
age               0
group             0
score             0
followup_score    1
dtype: int64


Equivalent pandas merge: Left join

In [27]:
connection.close()

# HTTP

In [28]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(
    url,
    timeout=10
)

In [29]:
response.status_code

200

In [30]:
response.status_code

200

The program should check status before analysis to make sure the data was actually collected from the site, or in some cases updated.

# JSON

In [31]:
response.text[:500]

'[\n  {\n    "id": 1,\n    "name": "Leanne Graham",\n    "username": "Bret",\n    "email": "Sincere@april.biz",\n    "address": {\n      "street": "Kulas Light",\n      "suite": "Apt. 556",\n      "city": "Gwenborough",\n      "zipcode": "92998-3874",\n      "geo": {\n        "lat": "-37.3159",\n        "lng": "81.1496"\n      }\n    },\n    "phone": "1-770-736-8031 x56442",\n    "website": "hildegard.org",\n    "company": {\n      "name": "Romaguera-Crona",\n      "catchPhrase": "Multi-layered client-server neural-'

In [32]:
data = response.json()

print(f"\n{type(data) = }\n")
print(f"\n{len(data) = }\n")
print(f"\n{type(data[0]) = }\n")
print(f"\n{data[0] = }\n")


type(data) = <class 'list'>


len(data) = 10


type(data[0]) = <class 'dict'>


data[0] = {'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}



The data from a JSON file becomes a list of python dictionaries.

In [33]:
users = pd.DataFrame(data)
print(f"{users.head()} \n")
print(f"{users.columns} \n")

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

         website                                            company  
0  hildegard.org  {'name': 'Romaguera-Crona', 'catchPhrase': 'Mu

In [34]:
users.loc[0, "address"]

{'street': 'Kulas Light',
 'suite': 'Apt. 556',
 'city': 'Gwenborough',
 'zipcode': '92998-3874',
 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}

The column contains a dictionary of the address values. The JSON structure had this dicitonary nested under the key "address", and it has not yet been flattened.

In [35]:
users_flat = pd.json_normalize(data)

In [36]:
print(f"\n{users_flat.head()}\n")
print(f"\n{users_flat.columns}")


   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburg

In [37]:
users_flat[[
"id",
"name",
"email",
"address.city",
"company.name"]]



,id,name,email,address.city,company.name
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy,Considine-Lockman
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth,Johns Group
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview,Abernathy Group
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury,Yost and Sons
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury,Hoeger LLC


In [38]:
json_path = RAW_DIR / "users.json"

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        data,
        f,
        indent=2
    )

Saving the response improves reproducibility because it allows others who see the code or analysis to use the same raw data when running their own analysis or reproducing the results from the original author's cleaning and analysis steps. It also preserves the version of the data used in the analysis, in case the original API endpoint changes what is available.

In [39]:
with open(
    json_path,
    "r",
    encoding="utf-8"
) as f:
    local_data = json.load(f)

local_data == data

True

In [40]:
bad_url = (
    "https://jsonplaceholder.typicode.com/"
    "does-not-exist"
)

bad_response = requests.get(
    bad_url,
    timeout=10
)

bad_response.status_code

404

In [ ]:
#bad_response.raise_for_status()

HTTPError: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/does-not-exist

Failure handling is part of data engineering because it affects what data is available to the analysis as well as how to collect the correct data in the quantity expected.

In [ ]:
# url = (
#     "https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States"
# )

# #%pip install lxml

# tables = pd.read_html(url)

# len(tables)
# tables[0].head()

HTTPError: HTTP Error 403: Forbidden

In [46]:
processed_csv = (
    PROCESSED_DIR / "users_processed.csv"
)

users_flat.to_csv(
    processed_csv,
    index=False
)

# Parquet

In [47]:
parquet_path = (
    PROCESSED_DIR / "users_processed.parquet"
)

users_flat.to_parquet(
    parquet_path,
    index=False
)

In [48]:
users_parquet = pd.read_parquet(
    parquet_path
)

print(f"{users_parquet.head()}\n")
print(f"{users_parquet.dtypes}\n")

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh

In [51]:
print(f"CSV: {processed_csv.stat().st_size}\n")
print(f"Parquet: {parquet_path.stat().st_size}\n")

CSV: 2434

Parquet: 11683



In [52]:
ratio = (
    parquet_path.stat().st_size
    / processed_csv.stat().st_size
)

ratio

4.799917830731307

Parquet is much larger than the CSV, but performance conclusions should not be generalized from such a small dataset.

In [53]:
csv_again = pd.read_csv(
    processed_csv
)

In [54]:
print(f"{csv_again.dtypes = }\n")
print(f"{users_parquet.dtypes = }\n")

csv_again.dtypes = id                       int64
name                       str
username                   str
email                      str
phone                      str
website                    str
address.street             str
address.suite              str
address.city               str
address.zipcode            str
address.geo.lat        float64
address.geo.lng        float64
company.name               str
company.catchPhrase        str
company.bs                 str
dtype: object

users_parquet.dtypes = id                     int64
name                     str
username                 str
email                    str
phone                    str
website                  str
address.street           str
address.suite            str
address.city             str
address.zipcode          str
address.geo.lat          str
address.geo.lng          str
company.name             str
company.catchPhrase      str
company.bs               str
dtype: object



A typed binary format has the advantage of storing the data types so they do not need to be explicitly stated each time the data is loaded.

# HDF5

In [55]:
hdf_path = (
    PROCESSED_DIR / "users.h5"
)

users_flat.to_hdf(
    hdf_path,
    key="users",
    mode="w"
)

In [57]:
users_hdf = pd.read_hdf(
    hdf_path,
    key="users"
)

users_hdf.head()

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


Specifying a key differs from reading a simple CSV because it gives 